# LLMs with Transformers and LangChain

This notebook provides a dive into the Hugging Face Transformers library with a focus on its usage for large language models (LLMs). We'll cover pipelines for inference, tokenization, model loading, and training with the Trainer API.

In [1]:
from typing import Dict, Any, Iterator

## Quick Tour: Transformers

In this section, we explore various functionalities of the Transformers library, including pipelines for inference, custom tokenization, and model usage across both PyTorch and TensorFlow frameworks.

### Pipeline

The [pipeline()](https://huggingface.co/docs/transformers/main/en/main_classes/pipelines#transformers.pipeline) is the easiest and fastest way to use a pretrained model for inference. You can use the [pipeline()](https://huggingface.co/docs/transformers/main/en/main_classes/pipelines#transformers.pipeline) out-of-the-box for many tasks across different modalities, some of which are shown in the table below:

<Tip>

For a complete list of available tasks, check out the [pipeline API reference](https://huggingface.co/docs/transformers/main/en/./main_classes/pipelines).

</Tip>

| **Task**                     | **Description**                                                                                              | **Modality**    | **Pipeline identifier**                       |
|------------------------------|--------------------------------------------------------------------------------------------------------------|-----------------|-----------------------------------------------|
| Text classification          | assign a label to a given sequence of text                                                                   | NLP             | pipeline(task=“sentiment-analysis”)           |
| Text generation              | generate text given a prompt                                                                                 | NLP             | pipeline(task=“text-generation”)              |
| Summarization                | generate a summary of a sequence of text or document                                                         | NLP             | pipeline(task=“summarization”)                |
| Image classification         | assign a label to an image                                                                                   | Computer vision | pipeline(task=“image-classification”)         |
| Image segmentation           | assign a label to each individual pixel of an image (supports semantic, panoptic, and instance segmentation) | Computer vision | pipeline(task=“image-segmentation”)           |
| Object detection             | predict the bounding boxes and classes of objects in an image                                                | Computer vision | pipeline(task=“object-detection”)             |
| Audio classification         | assign a label to some audio data                                                                            | Audio           | pipeline(task=“audio-classification”)         |
| Automatic speech recognition | transcribe speech into text                                                                                  | Audio           | pipeline(task=“automatic-speech-recognition”) |
| Visual question answering    | answer a question about the image, given an image and a question                                             | Multimodal      | pipeline(task=“vqa”)                          |
| Document question answering  | answer a question about a document, given an image and a question                                            | Multimodal      | pipeline(task="document-question-answering")  |
| Image captioning             | generate a caption for a given image                                                                         | Multimodal      | pipeline(task="image-to-text")                |

Start by creating an instance of [pipeline()](https://huggingface.co/docs/transformers/main/en/main_classes/pipelines#transformers.pipeline) and specifying a task you want to use it for. In this guide, you'll use the [pipeline()](https://huggingface.co/docs/transformers/main/en/main_classes/pipelines#transformers.pipeline) for sentiment analysis as an example:

#### Pipeline for Sentiment Analysis

Here, we create a sentiment analysis pipeline.

In [2]:
from transformers import pipeline

# Create a sentiment analysis pipeline using the default pretrained model.
sentiment_classifier = pipeline("sentiment-analysis")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps:0


In [3]:
result = sentiment_classifier("We are very happy to show you the 🤗 Transformers library.")[0]
print("Sentiment Analysis Result:", result)

Sentiment Analysis Result: {'label': 'POSITIVE', 'score': 0.9997795224189758}


For multiple inputs, simply pass a list of strings to the classifier to return a list of dictionaries

In [4]:
texts = [
    "We are very happy to show you the 🤗 Transformers library.",
    "We hope you don't hate it."
]

results = sentiment_classifier(texts)
for res in results:
    print(f"Label: {res['label']}, Score: {round(res['score'], 4)}")

Label: POSITIVE, Score: 0.9998
Label: NEGATIVE, Score: 0.5309


#### Pipeline for Automatic Speech Recognition (ASR)

Next, we set up a pipeline for automatic speech recognition using a pretrained model.

In [5]:
# Create an automatic speech recognition pipeline using the Facebook wav2vec2 model.
speech_recognizer = pipeline("automatic-speech-recognition", model="facebook/wav2vec2-base-960h", device='cpu')

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cpu


Load an audio dataset. Here we use the [PolyAI/minds14](https://huggingface.co/datasets/PolyAI/minds14) dataset, which will be automatically resampled to match the model's requirements.

In [6]:
import datasets
from datasets import load_dataset, Audio

# Load the dataset (using the "en-US" configuration and the "train" split).
dataset = load_dataset("PolyAI/minds14", name="en-US", split="train")

# Ensure the dataset's sampling rate matches the model's feature extractor sampling rate.
dataset = dataset.cast_column("audio", Audio(sampling_rate=speech_recognizer.feature_extractor.sampling_rate))

Extract the raw audio waveform arrays from the first four samples and pass them to the speech recognizer.

In [7]:
# Perform automatic speech recognition on the first 4 audio samples.
asr_results = speech_recognizer(dataset[:4]["audio"])
recognized_texts = [d["text"] for d in asr_results]

print("ASR Results:")
for text in recognized_texts:
    print(text)

ASR Results:
I WOULD LIKE TO SET UP A JOINT ACCOUNT WITH MY PARTNER HOW DO I PROCEED WITH DOING THAT
FONDERING HOW I'D SET UP A JOIN TO HELL T WITH MY WIFE AND WHERE THE AP MIGHT BE
I I'D LIKE TOY SET UP A JOINT ACCOUNT WITH MY PARTNER I'M NOT SEEING THE OPTION TO DO IT ON THE APSO I CALLED IN TO GET SOME HELP CAN I JUST DO IT OVER THE PHONE WITH YOU AND GIVE YOU THE INFORMATION OR SHOULD I DO IT IN THE AP AN I'M MISSING SOMETHING UQUETTE HAD PREFERRED TO JUST DO IT OVER THE PHONE OF POSSIBLE THINGS
HOW DO I FURN A JOINA COUT


For large datasets, rather than loading all audio samples into memory, we can define a generator that yields each sample (or batches of samples) one by one.
This snippet demonstrates a generator function that extracts the "audio" column from each dataset sample,
passes the yielded samples to the `speech_recognizer` pipeline, and prints the recognized texts from the first four samples.

In [8]:
def audio_samples_generator(dataset: datasets.Dataset, start: int = 0, end: int = None) -> Iterator[Any]:
    """
    Generator function to yield audio waveform arrays from a dataset.

    Parameters:
        dataset (Dataset): Hugging Face dataset that includes an 'audio' column.
        start (int): Starting index for dataset samples.
        end (int or None): Ending index for dataset samples. If None, yield until the end of the dataset.

    Yields:
        Any: The raw audio waveform array from each sample's 'audio' column.
    """
    if end is None:
        end = len(dataset)
    for i in range(start, end):
        yield dataset[i]["audio"]

# Create a generator for the first 4 samples.
audio_gen = audio_samples_generator(dataset, start=0, end=4)

# Pass the generator to the speech recognizer pipeline.
asr_results = speech_recognizer(audio_gen)

# Extract recognized texts from the ASR results.
recognized_texts = [result["text"] for result in asr_results]

print("ASR Results (using generator):")
for text in recognized_texts:
    print(text)

ASR Results (using generator):
I WOULD LIKE TO SET UP A JOINT ACCOUNT WITH MY PARTNER HOW DO I PROCEED WITH DOING THAT
FONDERING HOW I'D SET UP A JOIN TO HELL T WITH MY WIFE AND WHERE THE AP MIGHT BE
I I'D LIKE TOY SET UP A JOINT ACCOUNT WITH MY PARTNER I'M NOT SEEING THE OPTION TO DO IT ON THE APSO I CALLED IN TO GET SOME HELP CAN I JUST DO IT OVER THE PHONE WITH YOU AND GIVE YOU THE INFORMATION OR SHOULD I DO IT IN THE AP AN I'M MISSING SOMETHING UQUETTE HAD PREFERRED TO JUST DO IT OVER THE PHONE OF POSSIBLE THINGS
HOW DO I FURN A JOINA COUT


#### Using a Custom Model and Tokenizer in the Pipeline

You can use any model from the Hugging Face Hub for a specific use-case. For example, to perform sentiment analysis on French text, we load the multilingual BERT model.

In [9]:
model_name = "nlptown/bert-base-multilingual-uncased-sentiment"

# Load a pretrained model and tokenizer using the Auto classes.
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Use the loaded model and tokenizer in a pipeline for sentiment analysis.

In [10]:
# Create a pipeline using the custom model and tokenizer.
custom_classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)
french_text = "Nous sommes très heureux de vous présenter la bibliothèque 🤗 Transformers."
print("Custom Classifier Result:", custom_classifier(french_text))

Device set to use mps:0


Custom Classifier Result: [{'label': '5 stars', 'score': 0.7272652387619019}]


### AutoClass

An [AutoClass](https://huggingface.co/docs/transformers/main/en/./model_doc/auto) is a shortcut that automatically retrieves the architecture of a pretrained model from its name or path. It is only required to select the appropriate `AutoClass` for a task and it's associated preprocessing class. 

Let's return to the example from the previous section and see how you can use the `AutoClass` to replicate the results of the [pipeline()](https://huggingface.co/docs/transformers/main/en/main_classes/pipelines#transformers.pipeline).


#### AutoTokenizer

A tokenizer is responsible for preprocessing text into an array of numbers as inputs to a model. There are multiple rules that govern the tokenization process, including how to split a word and at what level words should be split (learn more about tokenization in the [tokenizer summary](https://huggingface.co/docs/transformers/main/en/./tokenizer_summary)). The most important thing to remember is you need to instantiate a tokenizer with the same model name to ensure you're using the same tokenization rules a model was pretrained with.

Load a tokenizer with [AutoTokenizer](https://huggingface.co/docs/transformers/main/en/model_doc/auto#transformers.AutoTokenizer):

In [11]:
from transformers import AutoTokenizer

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [12]:
# Tokenize a sample text using the loaded tokenizer.
encoding = tokenizer("We are very happy to show you the 🤗 Transformers library.")
print("Encoding Result:", encoding)

Encoding Result: {'input_ids': [101, 11312, 10320, 12495, 19308, 10114, 11391, 10855, 10103, 100, 58263, 13299, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


The tokenizer returns a dictionary containing:

* [input_ids](https://huggingface.co/docs/transformers/main/en/./glossary#input-ids): numerical representations of your tokens.
* [attention_mask](https://huggingface.co/docs/transformers/main/en/.glossary#attention-mask): indicates which tokens should be attended to.

A tokenizer can also accept a list of inputs, and pad and truncate the text to return a batch with uniform length:

In [13]:
batch = tokenizer(
    ["We are very happy to show you the 🤗 Transformers library.", "We hope you don't hate it."],
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt",
)

#### AutoModel

🤗 Transformers provides a simple and unified way to load pretrained instances. This means you can load an [AutoModel](https://huggingface.co/docs/transformers/main/en/model_doc/auto#transformers.AutoModel) like you would load an [AutoTokenizer](https://huggingface.co/docs/transformers/main/en/model_doc/auto#transformers.AutoTokenizer). The only difference is selecting the correct [AutoModel](https://huggingface.co/docs/transformers/main/en/model_doc/auto#transformers.AutoModel) for the task. For text (or sequence) classification, you should load [AutoModelForSequenceClassification](https://huggingface.co/docs/transformers/main/en/model_doc/auto#transformers.AutoModelForSequenceClassification):

In [14]:
from torch import nn
from transformers import AutoModelForSequenceClassification

# Load a model for sequence classification.
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Unpack and pass directly to the model the preprocessed batch of inputs
outputs = model(**batch)

# The model outputs the final activations in the `logits` attribute. Apply the softmax function to the `logits` to retrieve the probabilities:
probabilities = nn.functional.softmax(outputs.logits, dim=-1)
print("Predictions:\n", probabilities)

Predictions:
 tensor([[0.0021, 0.0018, 0.0115, 0.2121, 0.7725],
        [0.2084, 0.1826, 0.1969, 0.1755, 0.2365]], grad_fn=<SoftmaxBackward0>)


### Trainer API – A PyTorch Optimized Training Loop

The [Trainer](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer) class simplifies training for PyTorch models by wrapping the training loop and providing additional functionalities (e.g., distributed training and mixed precision).

For a typical training workflow, you will need:

1. [PreTrainedModel](https://huggingface.co/docs/transformers/main/en/main_classes/model#transformers.PreTrainedModel) or a [`torch.nn.Module`](https://pytorch.org/docs/stable/nn.html#torch.nn.Module).
2. [TrainingArguments](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.TrainingArguments) contains the model hyperparameters you can change like learning rate, batch size, and the number of epochs to train for. The default values are used if you don't specify any training arguments.
3. A preprocessing class like a tokenizer, image processor, feature extractor, or processor.
4. A dataset.
5. A [DataCollatorWithPadding](https://huggingface.co/docs/transformers/main/en/main_classes/data_collator#transformers.DataCollatorWithPadding) to create a batch of examples from the dataset.

```python

# Load a model
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased")

# Define training arguments
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="path/to/save/folder/",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    evaluation_strategy="steps"
)

# Load a tokenizer
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Load a dataset
from datasets import load_dataset
dataset = load_dataset("rotten_tomatoes")

# Create a function to tokenize the dataset and apply it over the entire dataset
def tokenize_dataset(dataset):
    return tokenizer(dataset["text"])

dataset = dataset.map(tokenize_dataset, batched=True)

# Data collator to dynamically pad inputs.
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
```

Now gather all these classes in [Trainer](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer):

```python
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)
```

When you're ready, call [train()](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer.train) to start training:
```python
trainer.train()
```

## LangChain

![](https://i.ibb.co/w6rhfn8/rag.webp)

In this section, we integrate LangChain with a HuggingFace model to build a question-answering chain.

You can use it for creation of
- **Chatbots**: Incorporates memory.
- **Agents**: Interacts with external tools.
- **Retrieval Augmented Generation (RAG) Apps**: Extracts information from internal resourses (knowledge graphs, etc.).

In [15]:
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

Where to look for models?

- https://lmarena.ai/?leaderboard
- https://huggingface.co/models?pipeline_tag=text-generation&sort=trending

In [16]:
# Define the model identifier for the HuggingFace model
# MODEL_ID: str = "gpt2"
MODEL_ID: str = "Qwen/Qwen2-0.5B-Instruct"

# Define pipeline
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
generation_pipeline = pipeline("text-generation", 
                model=model, 
                tokenizer=tokenizer, 
                max_new_tokens=100  # Increasing max_new_tokens may yield more detailed responses.
                )

llm_pipeline = HuggingFacePipeline(pipeline=generation_pipeline)

# Wrap the HuggingFace pipeline with LangChain's wrapper.
from langchain_huggingface import ChatHuggingFace
chat_model = ChatHuggingFace(llm=llm_pipeline)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
Device set to use mps:0


### Basic Chat Interaction and Message Persistence

In [17]:
from langchain_core.messages import (
    HumanMessage,
    AIMessage
)

In [18]:
# Single-turn invocation.
initial_response = chat_model.invoke([HumanMessage(content="Hi! I'm Bob")])
print("Initial Chat Response:")
initial_response.pretty_print()

Initial Chat Response:
================================== Ai Message ==================================

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hi! I'm Bob<|im_end|>
<|im_start|>assistant
Hello, Bob! How can I assist you today?


In [19]:
# Follow-up message without context (the model does not persist previous turns automatically).
followup_response = chat_model.invoke([HumanMessage(content="What's my name?")])
print("Follow-Up Chat Response (without context):")
followup_response.pretty_print()

Follow-Up Chat Response (without context):
================================== Ai Message ==================================

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What's my name?<|im_end|>
<|im_start|>assistant
Your name is your unique identifier and it is what people remember you as. It can be anything from "John Doe" to "Jane Smith". You can also use a nickname, which is just a short form of your full name. Your name is important because it tells other people who you are and what you do.


In [20]:
# Invoking the chat model with full conversation history to include context.
conversation_history = [
    HumanMessage(content="Hi! I'm Bob"),
    AIMessage(content="Hello Bob! How can I assist you today?"),
    HumanMessage(content="What's my name?")
]
contextual_response = chat_model.invoke(conversation_history)
print("Conversation with Context:")
contextual_response.pretty_print()

Conversation with Context:
================================== Ai Message ==================================

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hi! I'm Bob<|im_end|>
<|im_start|>assistant
Hello Bob! How can I assist you today?<|im_end|>
<|im_start|>user
What's my name?<|im_end|>
<|im_start|>assistant
Your name is Bob. Is there anything specific you would like to know or discuss about yourself?


### Integrate LangGraph for Conversation Persistence

In [21]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph

In [22]:
def call_model(state: MessagesState) -> dict:
    """
    Invoke the chat model on the provided conversation state and update the message history.
    
    Parameters:
        state (MessagesState): A dictionary with key 'messages' representing conversation history.
        
    Returns:
        dict: Updated conversation state with the latest AI response in 'messages'.
    """
    response = chat_model.invoke(state["messages"])
    return {"messages": response}

In [23]:
# Create a conversation graph using LangGraph with the state schema.
workflow = StateGraph(state_schema=MessagesState)
workflow.add_edge(START, "chat_model")
workflow.add_node("chat_model", call_model)

# Add in-memory persistence.
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)


# Define configuration with a unique thread identifier.
config = {"configurable": {"thread_id": "abc123"}}

# Simulate conversation: initial greeting.
query = "Hi! I'm Bob."
input_messages = [HumanMessage(content=query)]
output = app.invoke({"messages": input_messages}, config)
print("Chatbot Response to Initial Greeting:")
output["messages"][-1].pretty_print()

Chatbot Response to Initial Greeting:
================================== Ai Message ==================================

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hi! I'm Bob.<|im_end|>
<|im_start|>assistant
Hello, Bob! How can I assist you today?


In [24]:

# Simulate conversation: follow-up question.
query = "What's my name?"
input_messages = [HumanMessage(content=query)]
output = app.invoke({"messages": input_messages}, config)
print("Chatbot Response to Follow-Up Question:")
output["messages"][-1].pretty_print()

Chatbot Response to Follow-Up Question:
================================== Ai Message ==================================

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hi! I'm Bob.<|im_end|>
<|im_start|>assistant
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hi! I'm Bob.<|im_end|>
<|im_start|>assistant
Hello, Bob! How can I assist you today?<|im_end|>
<|im_start|>user
What's my name?<|im_end|>
<|im_start|>assistant
Your name is Bob. Is there something specific you would like to know or talk about?


### Incorporate Prompt Templates into the Conversation

**Prompt Templates** help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [25]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [26]:
# Define a ChatPromptTemplate with a system instruction and a placeholder for conversation messages.
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You talk like a pirate. Answer all questions to the best of your ability."),
    MessagesPlaceholder(variable_name="messages"),
])

In [27]:
def call_model_with_prompt(state: MessagesState) -> dict:
    """
    Invoke the chat model using a custom prompt template and update conversation history.
    
    Parameters:
        state (MessagesState): A dictionary with key 'messages' for the conversation history.
        
    Returns:
        dict: Updated conversation state with the AI response.
    """
    prompt = prompt_template.invoke(state)
    response = chat_model.invoke(prompt)
    return {"messages": response}

# Create a new workflow that incorporates the prompt template.
workflow_with_prompt = StateGraph(state_schema=MessagesState)
workflow_with_prompt.add_edge(START, "chat_model")
workflow_with_prompt.add_node("chat_model", call_model_with_prompt)

# Compile with memory persistence.
memory = MemorySaver()
app_with_prompt = workflow_with_prompt.compile(checkpointer=memory)

In [28]:
# Simulate conversation using the custom prompt.
config = {"configurable": {"thread_id": "abc345"}}
query = "Hi! I'm Jim."
input_messages = [HumanMessage(content=query)]
output = app_with_prompt.invoke({"messages": input_messages}, config)
print("Chatbot Response with Custom Prompt:")
output["messages"][-1].pretty_print()

Chatbot Response with Custom Prompt:
================================== Ai Message ==================================

<|im_start|>system
You talk like a pirate. Answer all questions to the best of your ability.<|im_end|>
<|im_start|>user
Hi! I'm Jim.<|im_end|>
<|im_start|>assistant
Hello, Jim! Nice to meet you. How can I assist you today?


In [29]:
query = "What is my name?"
input_messages = [HumanMessage(content=query)]
output = app_with_prompt.invoke({"messages": input_messages}, config)
print("Follow-Up Chatbot Response with Context:")
output["messages"][-1].pretty_print()

Follow-Up Chatbot Response with Context:
================================== Ai Message ==================================

<|im_start|>system
You talk like a pirate. Answer all questions to the best of your ability.<|im_end|>
<|im_start|>user
Hi! I'm Jim.<|im_end|>
<|im_start|>assistant
<|im_start|>system
You talk like a pirate. Answer all questions to the best of your ability.<|im_end|>
<|im_start|>user
Hi! I'm Jim.<|im_end|>
<|im_start|>assistant
Hello, Jim! Nice to meet you. How can I assist you today?<|im_end|>
<|im_start|>user
What is my name?<|im_end|>
<|im_start|>assistant
Your name is Jim.


### Extend the Conversation State with Additional Parameters

In [30]:
from typing import Sequence

from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from typing_extensions import Annotated, TypedDict

In [31]:
# Define a TypedDict to include conversation messages and a language parameter.
class ConversationState(TypedDict):
    """
    Represents the conversation state with messages and language.
    
    Attributes:
        messages (Sequence[BaseMessage]): Conversation history.
        language (str): Language specification for prompt customization.
    """
    messages: Annotated[Sequence[BaseMessage], add_messages]
    language: str

In [32]:
# Create a ChatPromptTemplate that includes a language variable.
prompt_template_with_language = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer all questions to the best of your ability in {language}."),
    MessagesPlaceholder(variable_name="messages"),
])

def call_model_with_language(state: ConversationState) -> dict:
    """
    Invoke the chat model using a prompt template that includes a language parameter.
    
    Parameters:
        state (ConversationState): Dictionary with 'messages' and 'language'.
        
    Returns:
        dict: Updated conversation state including the latest AI response.
    """
    prompt = prompt_template_with_language.invoke(state)
    response = chat_model.invoke(prompt)
    return {"messages": [response]}

# Create a workflow using the extended conversation state.
workflow_with_language = StateGraph(state_schema=ConversationState)
workflow_with_language.add_edge(START, "chat_model")
workflow_with_language.add_node("chat_model", call_model_with_language)

# Compile with persistence.
memory = MemorySaver()
app_with_language = workflow_with_language.compile(checkpointer=memory)

In [33]:
# Demonstrate conversation with a language parameter.
config = {"configurable": {"thread_id": "abc456"}}
query = "Hi! I'm Bob."
language = "Spanish"
input_messages = [HumanMessage(content=query)]
output = app_with_language.invoke(
    {"messages": input_messages, "language": language},
    config,
)
print("Chatbot Response with Language Parameter:")
output["messages"][-1].pretty_print()

Chatbot Response with Language Parameter:
================================== Ai Message ==================================

<|im_start|>system
You are a helpful assistant. Answer all questions to the best of your ability in Spanish.<|im_end|>
<|im_start|>user
Hi! I'm Bob.<|im_end|>
<|im_start|>assistant
¡Hola, Bob! ¿Cómo puedo ayudarte hoy?


In [34]:
# Follow-up conversation (language persists in the state).
query = "What is my name?"
input_messages = [HumanMessage(content=query)]
output = app_with_language.invoke(
    {"messages": input_messages},
    config,
)
print("Follow-Up Chatbot Response with Persisted Language:")
output["messages"][-1].pretty_print()

Follow-Up Chatbot Response with Persisted Language:
================================== Ai Message ==================================

<|im_start|>system
You are a helpful assistant. Answer all questions to the best of your ability in Spanish.<|im_end|>
<|im_start|>user
Hi! I'm Bob.<|im_end|>
<|im_start|>assistant
<|im_start|>system
You are a helpful assistant. Answer all questions to the best of your ability in Spanish.<|im_end|>
<|im_start|>user
Hi! I'm Bob.<|im_end|>
<|im_start|>assistant
¡Hola, Bob! ¿Cómo puedo ayudarte hoy?<|im_end|>
<|im_start|>user
What is my name?<|im_end|>
<|im_start|>assistant
Tu nombre es Bob.


### Manage Conversation History with Message Trimming

In [35]:
from langchain_core.messages import SystemMessage, trim_messages

In [36]:
# Create an instance of message trimmer to constrain the number of tokens.
trimmer = trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=chat_model,
    include_system=True,
    allow_partial=False,
    start_on="human",
)

# Example conversation messages (for demonstration purposes).
example_messages = [
    SystemMessage(content="You're a good assistant."),
    HumanMessage(content="Hi! I'm Bob."),
    AIMessage(content="Hi!"),
    HumanMessage(content="I like vanilla ice cream."),
    AIMessage(content="Nice."),
    HumanMessage(content="What's 2 + 2?"),
    AIMessage(content="4."),
    HumanMessage(content="Thanks!"),
    AIMessage(content="No problem!"),
    HumanMessage(content="Having fun?"),
    AIMessage(content="Yes!"),
]

print("Original number of messages:", len(example_messages))
trimmed_messages = trimmer.invoke(example_messages)
print("Number of messages after trimming:", len(trimmed_messages))

Original number of messages: 11
Number of messages after trimming: 7


In [37]:
def call_model_with_trimming(state: ConversationState) -> dict:
    """
    Invoke the chat model after trimming the conversation history to meet token constraints.
    
    Parameters:
        state (ConversationState): Dictionary with 'messages' and 'language'.
        
    Returns:
        dict: Updated conversation state including the latest model response.
    """
    # Trim the conversation history.
    trimmed_messages = trimmer.invoke(state["messages"])
    # Generate prompt using the trimmed messages and language.
    prompt = prompt_template_with_language.invoke({
        "messages": trimmed_messages,
        "language": state["language"]
    })
    response = chat_model.invoke(prompt)
    return {"messages": [response]}

# Create a new workflow that incorporates message trimming.
workflow_with_trimming = StateGraph(state_schema=ConversationState)
workflow_with_trimming.add_edge(START, "chat_model")
workflow_with_trimming.add_node("chat_model", call_model_with_trimming)

# Compile the workflow with persistence.
memory = MemorySaver()
app_with_trimming = workflow_with_trimming.compile(checkpointer=memory)

In [38]:
# Demonstrate conversation using message trimming.
config = {"configurable": {"thread_id": "abc123"}}
query = "Hi! I'm Bob."
language = "Spanish"
input_messages = [HumanMessage(content=query)]
output = app_with_trimming.invoke(
    {"messages": input_messages, "language": language},
    config,
)
print("Chatbot Response with Message Trimming:")
output["messages"][-1].pretty_print()

Chatbot Response with Message Trimming:
================================== Ai Message ==================================

<|im_start|>system
You are a helpful assistant. Answer all questions to the best of your ability in Spanish.<|im_end|>
<|im_start|>user
Hi! I'm Bob.<|im_end|>
<|im_start|>assistant
¡Hola, Bob! ¿Cómo estás?


In [39]:
# Retrieve and display the persisted conversation state.
state = app_with_trimming.get_state(config).values
print(f"Persisted Language: {state['language']}")
print("Persisted Conversation History:")
for message in state["messages"]:
    message.pretty_print()

Persisted Language: Spanish
Persisted Conversation History:
================================ Human Message =================================

Hi! I'm Bob.
================================== Ai Message ==================================

<|im_start|>system
You are a helpful assistant. Answer all questions to the best of your ability in Spanish.<|im_end|>
<|im_start|>user
Hi! I'm Bob.<|im_end|>
<|im_start|>assistant
¡Hola, Bob! ¿Cómo estás?
